# Omni NCM v2 — Colab training (latency-optimised)

Gets the code two ways — clone `adi274903/ncm-omni-v2`, or extract an uploaded `ncm_omni_v2_src.tgz` when you have no push access — then produces a shard corpus and trains with
`torch.compile` + bf16 autocast + fused AdamW.

What the last optimisation pass changed, and why the numbers below are measurable:

| lever | effect |
|---|---|
| `torch.compile(model, dynamic=False)`, warmed before the timed loop | **1.64-1.73x** step time |
| sync-free `omni_loss` (no `tensor.any()` barriers mid-graph) | frees the forward to queue ahead of the host |
| cached level/rope/expert-routing statics + LRU mask cache | nothing rebuilt per step |
| `index_select` MoE routing instead of boolean-mask indexing | no dynamo graph break per block |
| fused AdamW, TF32, bf16 autocast | fewer launches, more tensor-core band |

Measured on an 8 GB RTX 3070 laptop, `md` preset (82.48M), 441 packed positions, strict arm, MoE:
**0.11 s/step, 33.5k pos/s, 16.6 TFLOPS** (a measured bf16 cuBLAS ceiling of 27 TFLOPS on that chip).

Read the `tflops` column, not the wall clock: it is `6*N*D` per step divided by elapsed time, so a
faster step moves it and a debugger breakpoint does not.


In [ ]:
# ---- env: Colab secrets first, then the process environment ----
# GITHUB_TOKEN is only needed for the git route. If you uploaded the source archive instead, skip it.
import os, sys, subprocess

def _secret(name):
    try:
        from google.colab import userdata           # noqa: PLC0415
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

GITHUB_TOKEN = _secret("GITHUB_TOKEN")              # optional: private clone from GitHub
HF_TOKEN = _secret("HF_TOKEN")                      # optional: faster/limed Hub reads
print("github token", "ok" if GITHUB_TOKEN else "absent (source-archive mode)",
      "| hf token", "ok" if HF_TOKEN else "absent")


In [ ]:
# ---- get the code: use the copy already open, else an uploaded archive, else clone ----
import glob, os, subprocess, sys

def _walk_up(start, marker=("omni", "model.py")):
    """The repo root is wherever omni/model.py lives above the working directory."""
    d = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(d, *marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent

ARCHIVES = sorted(glob.glob("/content/ncm_omni_v2*.tgz") + glob.glob("/content/ncm_omni_v2*.zip")
                  + glob.glob("/content/drive/MyDrive/ncm_omni_v2*.tgz"))

PKG = (_walk_up(os.getcwd())
       or ("/content/ncm_omni_v2" if os.path.isdir("/content/ncm_omni_v2") else None))
print("cwd", os.getcwd(), "-> code at", PKG)

if PKG is None and ARCHIVES:
    # no git, no credentials: an uploaded tarball is the source of truth for this session
    src = ARCHIVES[0]
    print("extracting", src)
    if src.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", src, "-d", "/content"], check=True)
    else:
        subprocess.run(["tar", "xzf", src, "-C", "/content"], check=True)
    PKG = _walk_up("/content") or "/content/ncm_omni_v2"

if PKG is None:
    token = globals().get("GITHUB_TOKEN") or ""
    assert token, ("open this notebook inside the repo, upload ncm_omni_v2_src.tgz to /content, "
                   "or add a GITHUB_TOKEN secret")
    askpass = "/tmp/_git_askpass.sh"
    with open(askpass, "w") as f:                   # the token never lands in argv or output
        f.write('#!/bin/sh\ncase "$1" in *sername*) echo x-access-token ;; *) echo %s ;; esac\n'
                % token)
    os.chmod(askpass, 0o700)
    env = dict(os.environ, GIT_ASKPASS=askpass, GIT_TERMINAL_PROMPT="0")
    r = subprocess.run(["git", "clone", "https://github.com/adi274903/ncm-omni-v2.git",
                        "/content/ncm_omni_v2"], env=env, capture_output=True, text=True)
    print(r.returncode, r.stderr[-400:])
    os.remove(askpass)
    PKG = "/content/ncm_omni_v2"

os.chdir(PKG); sys.path.insert(0, PKG)

# the producer decodes all three modalities: datasets/huggingface_hub for the streams, soundfile
# and av for audio, pillow for images. torch is left alone so the image's CUDA wheel survives.
req = os.path.join(PKG, "requirements.txt")
deps = ["-r", req] if os.path.exists(req) else [
    "numpy", "tqdm", "tensorboard", "datasets", "huggingface_hub", "soundfile", "av", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "-q", "install"] + deps, check=True)
print("cwd", os.getcwd(), "| deps", "requirements.txt" if deps[0] == "-r" else "explicit list")


In [ ]:
# ---- hardware gate: fail loudly here, not 40 minutes into a producer ----
import torch, sys
import omni.train as T

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"
p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 1e9
print(f"{p.name} | sm {torch.cuda.get_device_capability(0)} | {vram:.1f} GB")
print("fast math:", T.fast_math())

# 441 packed positions/sample at 82.48M params: 16 fits an 8 GB laptop, 32 a 16 GB T4, 64 an A100
BATCH = 16 if vram < 12 else (32 if vram < 20 else 64)
STEPS = 1500
print(f"batch {BATCH} | steps {STEPS} -> {BATCH * STEPS:,} samples seen once")


In [ ]:
# ---- invariants: must print PIPELINE CHECK OK ----
import run_omni
run_omni.main(["check", "--preset", "md", "--base-len", "64"])


In [ ]:
# ---- produce the shard corpus (this is the slow, network-bound part) ----
# 8 producers each read a disjoint slice of every Hub stream. --max-ahead lifts the rolling
# buffer cap, which otherwise blocks forever in batch mode because nothing deletes shards.
import os, subprocess, sys, time

assert "PKG" in globals(), "run the code cell above first"

def live(argv, head=""):
    print(f"{head}[{time.strftime('%H:%M:%S')}]", " ".join(argv), flush=True)
    p = subprocess.Popen(argv, cwd=PKG, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    print(f"[{time.strftime('%H:%M:%S')}] exit {p.returncode}")
    return p.returncode

live([sys.executable, "-u", "run_omni.py", "produce", "--budget", "20000",
      "--preset", "md", "--base-len", "64", "--workers", "8", "--foreground",
      "--max-ahead", "100000"], head="produce ")


In [ ]:
# ---- A/B the one thing that dwarfs everything else: compile on vs off ----
# Two 20-step smokes on the corpus just produced. Copy the tflops pair into your notes.
import glob, json, os

def last_metrics():
    # single-arm runs write run/<arm>/metrics.jsonl; duplex writes run/metrics.jsonl
    fs = sorted(glob.glob(os.path.join(PKG, "run", "**", "metrics.jsonl"), recursive=True),
                key=os.path.getmtime)
    if not fs:
        return None
    return json.loads(open(fs[-1]).readlines()[-1])

for label, extra in (("eager   ", ["--no-compile"]), ("compiled", [])):
    live([sys.executable, "-u", "run_omni.py", "train", "--arm", "block", "--preset", "md",
          "--base-len", "64", "--batch", str(BATCH), "--smoke", "20",
          "--no-preflight"] + extra, head=label + " ")
    m = last_metrics()
    print(f"  -> {label}: {m['pos_per_s']:,.0f} pos/s | {m['tflops']} TFLOPS", flush=True)


In [ ]:
# ---- the run ----
# --save-every 200 is the only defence against a Colab disconnect: checkpoints land in
# run/train_block/. Mount Drive first if the session is short.
import os
live([sys.executable, "-u", "run_omni.py", "train", "--arm", "block", "--preset", "md",
      "--base-len", "64", "--batch", str(BATCH), "--steps", str(STEPS),
      "--eval-every", "500", "--save-every", "200", "--log-every", "10"],
     head="train ")


In [ ]:
# ---- held-out eval + a progress table from metrics.jsonl ----
import glob, json, os

live([sys.executable, "-u", "run_omni.py", "eval", "--preset", "md", "--base-len", "64"],
     head="eval ")

for f in sorted(glob.glob(os.path.join(PKG, "run", "eval*.json"))):
    d = json.load(open(f))
    print("==", os.path.basename(f))
    print(json.dumps(d, indent=1)[:1800])

rows = [json.loads(l) for l in open(os.path.join(PKG, "run", "train_block", "metrics.jsonl"))]
step = max(1, len(rows) // 8)
print("\n step |   loss |  pos/s | TFLOPS")
for r in rows[::step] + [rows[-1]]:
    print(f"{r['step']:5d} | {r['loss']:6.3f} | {r['pos_per_s']:6.0f} | {r['tflops']:6.1f}")


## Optional — the `strict` arm, matched against flat `ar`

`--arms strict,ar --live --inproc` streams from the Hub directly and steps **both arms on the
identical batch**, so the comparison is structural. It is also ~1.5 samples/s, i.e. a sanity check,
not a training run. Skip it unless you are testing the level-parallel claim itself.

```python
run_omni.main(["train", "--arms", "strict,ar", "--inproc", "--live", "--preset", "md",
               "--base-len", "64", "--batch", str(BATCH), "--steps", "4", "--log-every", "1"])
```

## Reading the result

`tflops` is the throughput claim; `loss` per level/modality is the learning claim. Only the
**conditional** eval numbers carry weight — free-running is structurally void because feeding the
model its own codes compounds one bad level into every coarser one.

At a few hundred steps and 0.2 positions/parameter both arms are undertrained: a gap is a
direction, not a verdict. The 8 GB laptop measured 16.6 TFLOPS at batch 8 / 18.8 at batch 16 /
21.4 with the 8180-way finest head; if Colab reports less than ~15 TFLOPS on a T4, check that
`torch.compile` actually engaged (the startup line prints graphs and compile seconds) and that the
runtime is a GPU one.

Checkpoints: `run/train_block/step_*.pt` (each carries its own cfg, so `omni.eval` reloads the exact
model). Copy them to Drive before the session dies.
